# PT04: Convolutional neural networks from scratch

In this lab we implement the basic building blocks of convolutional neural networks (convolutional layers and pooling) from scratch in PyTorch, and we use them to classify cursive Japanese characters. All of these can be found (already implemented) in the `torch.nn` module, making the lab highly didactical (but very informative on what a convolution really computes, and on how modules work in PyTorch). PT05, after the lecture on deeper models, continues with batch normalization, residual blocks and dropout.

| Activity | What you write | Section | |
|---|---|---|---|
| 1 | A convolutional layer with `unfold` | 3 | ★★ |
| 2 | Max-pooling | 4 | ★ |
| 3 | A complete CNN | 5 | ★ |
| 4 | An occlusion map | 6 | ★★ |

Exercises are rated from 1 to 3 stars based on expected difficulty. Seven optional sections close the notebook: stride, depthwise convolutions, translation equivariance, a locally-connected layer, patches without copies, receptive fields and causal convolutions.

**How to work.** Run the cells in order. Each activity ends with a one-line check that stops with a message telling you what to look at. A function you have not written yet raises `NotImplementedError` the first time it is used, so complete the activities in order. Data loading and the training loop are given. The solutions, with a short explanation for each exercise, are in [PT04 solutions](https://colab.research.google.com/github/sscardapane/teaching-materials/blob/main/notebooks/PT04_Convolutional_neural_networks_solutions.ipynb).

We assume PT01 and PT02 (tensors, autograd and the training loop). Everything runs on the CPU in a few minutes; in Colab, a GPU runtime makes training faster but is not needed.

> Try to solve these exercises without AI coding assistants.

## 1. Setup and data

We use [jaxtyping](https://docs.kidger.site/jaxtyping/) to annotate shapes, as in PT02, and [torchvision](https://docs.pytorch.org/vision/stable/index.html) for loading and preprocessing the data. The second code cell defines the checks of the activities.

In [ ]:
%pip install jaxtyping --quiet

In [ ]:
import math
import time
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from torch.nn import Parameter
from torchvision.transforms import v2
from jaxtyping import Float

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False)
print("PyTorch", torch.__version__, "· torchvision", torchvision.__version__)

In [ ]:
#@title Checks for the activities (run this cell; you do not need to read it)
def _compare(actual, expected, what):
    assert actual.shape == expected.shape, f"{what}: expected shape {tuple(expected.shape)}, got {tuple(actual.shape)}."
    assert torch.allclose(actual, expected, atol=1e-5, rtol=1e-4), f"{what}: wrong values."

def _conv_layer(make, C, C_out, s, **kwargs):
    layer = make(C, C_out, s, **kwargs)
    for name, shape in [("weight", (C_out, C, s, s)), ("bias", (C_out,))]:
        value = getattr(layer, name, None)
        assert isinstance(value, Parameter), f"{name} must be a Parameter, so that the optimizer finds it."
        assert value.shape == shape, f"{name} should have shape {shape}, as in torch.nn.Conv2d, got {tuple(value.shape)}."
    with torch.no_grad():
        layer.bias.normal_()  # A zero bias would hide a mistake in adding it.
    return layer

def check_conv(MyConv2d):
    torch.manual_seed(0)
    for C, C_out, H, W, s, p in [(3, 8, 10, 12, 3, 1), (1, 4, 9, 9, 5, 0)]:
        layer = _conv_layer(MyConv2d, C, C_out, s, padding=p)
        x = torch.randn(2, C, H, W)
        _compare(layer(x), F.conv2d(x, layer.weight, layer.bias, padding=p),
                 f"MyConv2d({C}, {C_out}, {s}, padding={p}) on an input of shape {(2, C, H, W)}")
    print("MyConv2d is correct.")

def check_pool(MyMaxPool2d):
    x = torch.randn(2, 3, 8, 8, generator=torch.Generator().manual_seed(1)) - 2  # Mostly negative values.
    _compare(MyMaxPool2d(2)(x), F.max_pool2d(x, 2), "MyMaxPool2d(2) on an input of shape (2, 3, 8, 8)")
    print("MyMaxPool2d is correct.")

def check_cnn(MyCNN):
    torch.manual_seed(0)
    model = MyCNN(10)
    containers = (torch.nn.ModuleList, torch.nn.Sequential, torch.nn.ModuleDict)
    for name, module in model.named_modules():
        if name and not isinstance(module, containers) and type(module).__module__.startswith("torch."):
            raise AssertionError(f"{name} is a {type(module).__name__} from torch.nn: use your own layers.")
    n = sum(p.numel() for p in model.parameters())
    assert 0 < n < 100_000, f"The model has {n:,} parameters; keep it below 100,000."
    out = model(torch.randn(64, 1, 28, 28))
    assert out.shape == (64, 10), f"Expected one logit per class, shape (64, 10), got {tuple(out.shape)}."
    print(f"MyCNN is correct: {n:,} parameters.")

def check_occlusion(occlusion_map, model):
    image, label = val_x[0], int(val_y[0])
    result = occlusion_map(model, image, label)
    assert result.shape == (8, 8), f"Expected shape (8, 8) for patch 7 and stride 3, got {tuple(result.shape)}."
    covered = image.clone()
    covered[:, 6:13, 3:10] = background
    with torch.no_grad():
        model.eval()
        expected = model(covered[None]).softmax(dim=1)[0, label]
    assert torch.allclose(result[2, 1], expected, atol=1e-5), "Row 2, column 1 should cover rows 6 to 12 and columns 3 to 9 with background."
    print("occlusion_map is correct.")

def check_strided_conv(MyStridedConv2d):
    torch.manual_seed(0)
    for C, C_out, H, W, s, stride, p in [(3, 8, 11, 10, 3, 2, 1), (2, 4, 9, 9, 5, 3, 0)]:
        layer = _conv_layer(MyStridedConv2d, C, C_out, s, stride=stride, padding=p)
        x = torch.randn(2, C, H, W)
        _compare(layer(x), F.conv2d(x, layer.weight, layer.bias, stride=stride, padding=p),
                 f"Kernel {s}, stride {stride}, padding {p} on an input of shape {(2, C, H, W)}")
    print("MyStridedConv2d is correct.")

def check_depthwise(MyDepthwiseConv2d):
    torch.manual_seed(0)
    layer = MyDepthwiseConv2d(4, 3, padding=1)
    assert layer.weight.shape == (4, 1, 3, 3) and layer.bias.shape == (4,), "weight should have shape (4, 1, 3, 3) and bias (4,)."
    with torch.no_grad():
        layer.bias.normal_()
    x = torch.randn(2, 4, 7, 9)
    _compare(layer(x), F.conv2d(x, layer.weight, layer.bias, padding=1, groups=4),
             "MyDepthwiseConv2d(4, 3, padding=1) on an input of shape (2, 4, 7, 9)")
    print("MyDepthwiseConv2d is correct.")

def check_equivariance_error(equivariance_error):
    x = torch.randn(2, 1, 10, 10, generator=torch.Generator().manual_seed(5))
    result = equivariance_error(lambda x: x, x, 2, 1, 0)
    assert isinstance(result, float), "Return a Python float, e.g. with .item()."
    assert result == 0, "The identity is equivariant everywhere: the error should be 0."
    assert equivariance_error(lambda x: x.flip(-1), x, 0, 2, 3) > 0.1, "Flipping the image is not equivariant: the error should be large."
    added = lambda x: x + 1  # The constant is missing in the strip emptied by the shift.
    assert equivariance_error(added, x, 3, 0, 0) == 1 and equivariance_error(added, x, 3, 0, 3) == 0, \
        "Crop `border` pixels on every side of both outputs before taking the maximum."
    print("equivariance_error is correct.")

def check_locally_connected(MyLocallyConnected2d, MyConv2d):
    local = MyLocallyConnected2d(2, 3, 3, height=6, width=7, padding=1)
    assert local.weight.shape == (42, 3, 18) and local.bias.shape == (42, 3), \
        "For a 6x7 input with padding 1 there are 42 windows: weight (42, 3, 18), bias (42, 3)."
    x = torch.randn(4, 2, 6, 7)
    conv = MyConv2d(2, 3, 3, padding=1)
    with torch.no_grad():  # With the same kernel in every window, the layer is a convolution.
        local.weight.copy_(conv.weight.view(1, 3, 18).expand(42, -1, -1))
        conv.bias.normal_()
        local.bias.copy_(conv.bias.expand(42, -1))
    _compare(local(x), conv(x), "With the same kernel in every window, the layer should match MyConv2d")
    with torch.no_grad():
        local.weight[10].normal_()
    changed = (local(x) - conv(x)).abs().sum(dim=(0, 1)).flatten().nonzero().squeeze(1).tolist()
    assert changed == [10], f"Changing the kernel of window 10 changed the output positions {changed}."
    print("MyLocallyConnected2d is correct.")

def check_patches_view(patches_view):
    x = torch.randn(2, 3, 6, 7)
    view = patches_view(x, 3)
    assert view.shape == (2, 3, 4, 5, 3, 3), f"Expected shape (2, 3, 4, 5, 3, 3), got {tuple(view.shape)}."
    assert view.untyped_storage().data_ptr() == x.untyped_storage().data_ptr(), "The result must be a view of x, not a copy."
    assert torch.equal(view[1, 2, 3, 1], x[1, 2, 3:6, 1:4]), "view[b, c, i, j] should be the 3x3 patch with top-left corner (i, j)."
    print("patches_view is correct.")

def check_receptive_field(receptive_field):
    for k in (3, 5):
        layer = torch.nn.Conv2d(1, 1, k, padding=k // 2)
        with torch.no_grad():
            layer.weight.uniform_(0.1, 1.)
        result = receptive_field(layer)
        assert tuple(result) == (k, k), f"A single {k}x{k} convolution has a receptive field of {(k, k)}, got {result}."
    print("receptive_field is correct.")

def check_causal_conv1d(causal_conv1d):
    gen = torch.Generator().manual_seed(10)
    x, weight, bias = torch.randn(2, 3, 12, generator=gen), torch.randn(4, 3, 5, generator=gen), torch.randn(4, generator=gen)
    _compare(causal_conv1d(x, weight, bias), F.conv1d(F.pad(x, (4, 0)), weight, bias),
             "causal_conv1d with kernel 5 on an input of shape (2, 3, 12)")
    print("causal_conv1d is correct.")

For this lab we use [KMNIST](https://github.com/rois-codh/kmnist), which is a simple drop-in replacement for MNIST. The input is a (28, 28) black-and-white image of a cursive Japanese character (*Kuzushiji*), which has to be classified into 10 different classes. There are 60,000 training images and 10,000 test images.

In [ ]:
train_raw = torchvision.datasets.KMNIST("./kmnist", train=True, download=True)
test_raw = torchvision.datasets.KMNIST("./kmnist", train=False, download=True)
classes = train_raw.classes
print(len(train_raw), "training images,", len(test_raw), "test images")

# A Dataset is fundamentally a sequence of (input, target) pairs.
image, label = train_raw[0]
print(type(image), label, classes[label])

fig, axes = plt.subplots(3, 10, figsize=(10, 3.4))
for c in range(10):
    examples = (train_raw.targets == c).nonzero().squeeze(1)[:3]
    for row, index in enumerate(examples):
        axes[row, c].imshow(train_raw.data[index], cmap="gray_r")
        axes[row, c].axis("off")
    axes[0, c].set_title(classes[c])
fig.tight_layout()
plt.show()

Transformations (see [here](https://docs.pytorch.org/vision/stable/transforms.html)) are applied when loading each image. Here we transform the PIL images into PyTorch tensors and standardize them, with a mean and standard deviation computed on the training images only. We keep 6,000 training images aside for validation. The test set is used only once, at the end of the lab.

In [ ]:
split = torch.randperm(60_000, generator=torch.Generator().manual_seed(0))
train_idx, val_idx = split[:54_000], split[54_000:]
pixels = train_raw.data[train_idx].float() / 255
mean, std = pixels.mean().item(), pixels.std().item()
print(f"Pixel mean {mean:.4f}, std {std:.4f} (training images)")

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((mean,), (std,)),
])
full_data = torchvision.datasets.KMNIST("./kmnist", train=True, transform=transform)
train_data = torch.utils.data.Subset(full_data, train_idx.tolist())
val_data = torch.utils.data.Subset(full_data, val_idx.tolist())
test_data = torchvision.datasets.KMNIST("./kmnist", train=False, transform=transform)

# Loaders are used to shuffle and batch the elements of the dataset.
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=500)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=500)

xb, yb = next(iter(train_loader))
# Shape is now (batch_size, channels, height, width), and the targets are (batch_size,).
print(xb.shape, yb.shape)
assert xb.shape == (64, 1, 28, 28) and yb.shape == (64,)
background = (0 - mean) / std  # The value of an empty pixel (0) after standardization.

The training loop is given. It is the mini-batch loop of PT02, with the AdamW optimizer and a validation pass at the end of every epoch. If a GPU is available, `fit` uses it and moves the model back to the CPU at the end.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on", device)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    model_device = next(model.parameters()).device
    total_loss, correct, count = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(model_device), y.to(model_device)
        logits = model(x)
        total_loss += F.cross_entropy(logits, y, reduction="sum").item()
        correct += (logits.argmax(dim=1) == y).sum().item()
        count += len(y)
    return total_loss / count, correct / count

def fit(model, loader=train_loader, epochs=3, lr=1e-3, verbose=True):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "val_accuracy": []}
    for epoch in range(epochs):
        model.train()
        start = time.time()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            optimizer.step()
            history["train_loss"].append(loss.item())
        val_loss, val_accuracy = evaluate(model, val_loader)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)
        if verbose:
            print(f"Epoch {epoch + 1}: validation loss {val_loss:.3f}, "
                  f"accuracy {val_accuracy:.3f} ({time.time() - start:.0f} s)")
    model.cpu()
    return history

def plot_histories(histories):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for name, history in histories.items():
        losses = torch.tensor(history["train_loss"])
        smooth = losses.unfold(0, 50, 50).mean(dim=1)  # Average of every 50 updates.
        axes[0].plot(torch.arange(len(smooth)) * 50, smooth, label=name)
        epochs = range(1, len(history["val_accuracy"]) + 1)
        axes[1].plot(epochs, history["val_accuracy"], "o-", label=name)
    axes[0].set(xlabel="Update", ylabel="Training loss", yscale="log")
    axes[1].set(xlabel="Epoch", ylabel="Validation accuracy")
    for ax in axes:
        ax.legend()
    fig.tight_layout()
    plt.show()

## 2. Layers are modules

PyTorch has many [pre-implemented layers](https://docs.pytorch.org/docs/stable/nn.html), such as a 2D convolutional layer.

**Predict.** The layer below has 1 input channel, 16 output channels, a 5×5 kernel, padding 2 and stride 2. What is the shape of `conv2d(xb)`? How many parameters does it have? Run the next cell to check.

In [ ]:
conv2d = torch.nn.Conv2d(1, 16, 5, padding=2, stride=2)
print(conv2d(xb).shape)
for name, parameter in conv2d.named_parameters():
    print(name, tuple(parameter.shape))
print("Parameters:", sum(p.numel() for p in conv2d.parameters()))

Most of these modules are subclasses of the [nn.Module](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) class, which specifies their initialization, their forward pass, their trainable (or non-trainable) parameters, etc. As a warm-up, here is a linear layer $f(\mathbf{X}) = \mathbf{X}\mathbf{W} + \mathbf{b}$ written from scratch. We will use it in the rest of the lab.

Two things to notice. Wrapping a tensor in a `Parameter` is what makes it appear in `model.parameters()`, so the optimizer can find it. And we choose the initialization ourselves: the weights are Gaussian with variance $2/d$ for $d$ inputs (*He initialization*), which keeps the scale of the activations roughly constant through ReLU layers.

In [ ]:
class MyLinear(torch.nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.weight = Parameter(torch.randn(in_features, out_features) * math.sqrt(2 / in_features))
        self.bias = Parameter(torch.zeros(out_features))

    def forward(self, xb: Float[torch.Tensor, "batch in_features"]) -> Float[torch.Tensor, "batch out_features"]:
        return xb @ self.weight + self.bias

linear = MyLinear(16, 32)
print(linear)
print([(name, tuple(p.shape)) for name, p in linear.named_parameters()])
assert linear(torch.randn(8, 16)).shape == (8, 32)

## 3. Activity 1: a convolutional layer ★★

Now let us re-implement the `Conv2d` module. In general, there are many ways to implement a convolution (e.g., with FFTs). In the lecture we saw a simple one: collect the flattened, padded $s \times s$ neighborhoods of every pixel as the columns of a matrix $U(X)$. The convolution is then a single matrix product:

$$\underbrace{\widetilde{Z}}_{(C',\,HW)} = \underbrace{W^\top}_{(C',\,s^2C)}\, \underbrace{U(X)}_{(s^2C,\,HW)} + \mathbf{b}.$$

In PyTorch, [unfold](https://docs.pytorch.org/docs/stable/generated/torch.nn.Unfold.html) builds $U(X)$ for a whole batch: for an input of shape $(B, C, H, W)$ it returns a tensor of shape $(B, C s^2, L)$, where $L$ is the number of sliding windows. Convolution is then matrix multiplication on these patches! Let us look at a tiny example first.

In [ ]:
tiny = torch.arange(1., 10.).view(1, 1, 3, 3)
print(tiny)
patches = F.unfold(tiny, kernel_size=2)
print(patches.shape)
print(patches)

There are four 2×2 windows, one per column, read row by row. Each column contains the four pixels of its window, again row by row. With more channels, the rows of each column are ordered by channel first, then by position in the window: this is the same order you get if you flatten a filter of shape $(C, s, s)$.

**Predict.** What is the shape of `F.unfold(xb, kernel_size=3, padding=1)`?

In [ ]:
print(F.unfold(xb, kernel_size=3, padding=1).shape)

> **TODO** (★★): using `unfold`, complete the layer below without any other tool from `torch.nn` apart from `Parameter`.

- `weight` must have shape `(out_channels, in_channels, kernel_size, kernel_size)` and `bias` shape `(out_channels,)`, as in `torch.nn.Conv2d`: the check copies your weights into PyTorch's function to compare the results. Initialize the weights with He initialization (here $d = C s^2$ inputs per output) and the bias with zeros.
- Support zero-padding, with stride 1. The output has height $H + 2p - s + 1$, and similarly for the width. (Optional A adds a stride.)

Hint 1: `self.weight.view(out_channels, -1)` flattens each filter in the same order used by `unfold`.

Hint 2: a matrix of shape $(C', s^2C)$ times a tensor of shape $(B, s^2C, L)$ broadcasts over the batch. After that, you only need to reshape $L$ back into the output height and width.

In [ ]:
class MyConv2d(torch.nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, padding: int = 0):
        super().__init__()
        # TODO: complete the code.
        raise NotImplementedError("Complete Activity 1: MyConv2d.__init__")

    def forward(self, xb: Float[torch.Tensor, "batch in_channels height width"]) -> Float[torch.Tensor, "batch out_channels new_height new_width"]:
        # TODO: complete the code.
        raise NotImplementedError("Complete Activity 1: MyConv2d.forward")

In [ ]:
check_conv(MyConv2d)

## 4. Activity 2: pooling ★

We now move to max-pooling. With your knowledge of `unfold`, this should be much easier.

> **TODO** (★): complete the custom implementation of max-pooling below, with non-overlapping windows of size `kernel_size` (as `torch.nn.MaxPool2d` does by default). You can assume that the height and width of the image are multiples of the window.

Hint: exploit the stride parameter of `unfold` to avoid overlap between the patches, and then apply the pooling operation on each patch. Be careful with the channels: unlike in a convolution, they are not mixed.

In [ ]:
class MyMaxPool2d(torch.nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        self.kernel_size = kernel_size

    def forward(self, xb: Float[torch.Tensor, "batch channels height width"]) -> Float[torch.Tensor, "batch channels reduced_height reduced_width"]:
        # TODO: complete the code. Note: reduced_height = height // kernel_size, and similarly for the width.
        raise NotImplementedError("Complete Activity 2: MyMaxPool2d.forward")

In [ ]:
check_pool(MyMaxPool2d)

The network below also uses *global average pooling*, which averages each channel over all positions, from $(B, C, H, W)$ to $(B, C)$. It is a single line:

In [ ]:
def global_avg_pool(xb: Float[torch.Tensor, "batch channels height width"]) -> Float[torch.Tensor, "batch channels"]:
    return xb.mean(dim=(2, 3))

**Predict.** Max-pooling has no parameters, but we still need to backpropagate through it. If we call `backward` on the sum of the pooled values, which entries of the input receive a nonzero gradient, and what is their value?

In [ ]:
x = torch.randn(1, 1, 4, 4, generator=torch.Generator().manual_seed(3), requires_grad=True)
MyMaxPool2d(2)(x).sum().backward()
print(x.detach().squeeze())
print(x.grad.squeeze())

## 5. Activity 3: building the network ★

Now for the fun part: assemble the previous blocks to build the final convolutional network. As an example, you could do:

x → [MyConv2d → ReLU → MyMaxPool2d] × 2 → MyConv2d → ReLU → global average pooling → MyLinear

> **TODO** (★): complete the specification of the model. Use only the layers you wrote in this notebook (plus `F.relu`), keep it below 100,000 parameters, and return one logit per class.

The check rejects layers from `torch.nn` (apart from containers such as `torch.nn.ModuleList` or `torch.nn.Sequential`). Feel free to experiment with the number of layers and channels.

In [ ]:
class MyCNN(torch.nn.Module):
    def __init__(self, n_classes: int = 10):
        super().__init__()
        # TODO: complete the code.
        raise NotImplementedError("Complete Activity 4: MyCNN.__init__")

    def forward(self, xb: Float[torch.Tensor, "batch channels height width"]) -> Float[torch.Tensor, "batch n_classes"]:
        # TODO: complete the code.
        raise NotImplementedError("Complete Activity 4: MyCNN.forward")

In [ ]:
check_cnn(MyCNN)

We now train the model, and we compare it with an MLP that receives the flattened image (a single hidden layer with 128 units). With the default settings, each epoch of the CNN takes about 20 seconds on a laptop CPU, and longer in a Colab CPU runtime.

In [ ]:
EPOCHS = 4
torch.manual_seed(0)
cnn = MyCNN(10)
cnn_history = fit(cnn, epochs=EPOCHS, lr=2e-3)

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, hidden: int = 128, n_classes: int = 10):
        super().__init__()
        self.hidden, self.out = MyLinear(28 * 28, hidden), MyLinear(hidden, n_classes)

    def forward(self, xb):
        return self.out(F.relu(self.hidden(xb.flatten(start_dim=1))))

torch.manual_seed(0)
mlp = MLP()
mlp_history = fit(mlp, epochs=EPOCHS, lr=2e-3)
plot_histories({"CNN": cnn_history, "MLP": mlp_history})
for name, model in [("CNN", cnn), ("MLP", mlp)]:
    print(f"{name}: {sum(p.numel() for p in model.parameters()):>7,} parameters, validation accuracy {evaluate(model, val_loader)[1]:.3f}")

**Discuss.** Which model is better on the validation set? Keep the answer in mind: we will look at the test set at the end of the lab.

## 6. Activity 4: what did the network learn? ★★

Let us look inside the trained CNN. The first row below shows the filters of the first layer, the second row the corresponding feature maps (after the ReLU) for one validation image.

In [ ]:
val_x, val_y = next(iter(torch.utils.data.DataLoader(val_data, batch_size=2000)))
with torch.no_grad():
    filters = cnn.conv1.weight[:, 0].detach()
    maps = F.relu(cnn.conv1(val_x[:1]))[0]
n_filters = min(16, len(filters))
fig, axes = plt.subplots(2, n_filters, figsize=(12, 1.8))
for i in range(n_filters):
    axes[0, i].imshow(filters[i], cmap="RdBu_r", vmin=-filters.abs().max(), vmax=filters.abs().max())
    axes[1, i].imshow(maps[i], cmap="gray_r")
for ax in axes.flat:
    ax.axis("off")
plt.show()

The 3×3 filters are hard to read on their own, but the feature maps show what they do: each one responds to the edges or strokes of the character in a different direction. Deeper layers are harder to visualize in this way. A simple alternative is an **occlusion map** (Zeiler and Fergus, 2014): cover a small square of the image with background, and see how much the probability of the correct class drops. If it drops a lot, the network was using that region.

> **TODO** (★★): complete `occlusion_map`. For every position of a `patch × patch` square, with top-left corners every `stride` pixels (`range(0, 28 - patch + 1, stride)` in both directions), replace the square with `background` and compute the probability that `model` assigns to `label`. Return a tensor of probabilities with one row per vertical position and one column per horizontal position.

Hint: build all the occluded images in a single batch and call the model once. Put the model in inference mode and do not track gradients.

In [ ]:
def occlusion_map(model, image: Float[torch.Tensor, "1 height width"], label: int, patch: int = 7, stride: int = 3) -> Float[torch.Tensor, "rows cols"]:
    # TODO: complete the code.
    raise NotImplementedError("Complete Activity 4: occlusion_map")

In [ ]:
check_occlusion(occlusion_map, cnn)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.2))
for n in range(5):
    image, label = val_x[n], int(val_y[n])
    heat = occlusion_map(cnn, image, label, patch=7, stride=1)
    axes[0, n].imshow(image[0], cmap="gray_r")
    axes[0, n].set_title(classes[label])
    axes[1, n].imshow(heat, cmap="magma", vmin=0, vmax=1)
    axes[1, n].set_title(f"min p = {heat.min():.2f}")
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()
plt.show()

**Discuss.** Dark regions are where an occlusion hurts the prediction. Are they on the strokes of the character, or somewhere else? Is the network relying on a single detail, or on the whole shape?

## 7. The test set

We chose all the settings above by looking at the validation set, so the test set gives an unbiased estimate for the final models.

In [ ]:
for name, model in [("CNN", cnn), ("MLP", mlp)]:
    print(f"{name}: validation accuracy {evaluate(model, val_loader)[1]:.3f}, test accuracy {evaluate(model, test_loader)[1]:.3f}")

**Discuss**

1. Both models lose accuracy on the test set, but not by the same amount. What does this suggest about the difference between the training and test images, and about which model generalizes better?
2. A CNN for 28×28 images with global average pooling can be applied to a 56×56 image without changing any weight. Why? What happens with an MLP?
3. In Activity 1, `unfold` creates a tensor $s^2$ times larger than the input. Why is this a problem for large images, and how do libraries avoid it? (See Optional E.)

This is the end of the core of the lab.

## Optional A: stride ★

The optional sections can be done in any order, except that Optional D uses your function from Optional C.

In the lecture, a convolution with stride $t$ keeps one window every $t$ pixels in each direction. The output then has height $\lfloor (H + 2p - s)/t \rfloor + 1$, and similarly for the width.

> **TODO** (★): complete `MyStridedConv2d`, a subclass of your `MyConv2d` with a `stride` argument. It reuses the weights and the bias of `MyConv2d`, so you only need to write `forward`. `F.unfold` accepts a `stride` argument.

In [ ]:
class MyStridedConv2d(MyConv2d):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int = 1, padding: int = 0):
        super().__init__(in_channels, out_channels, kernel_size, padding=padding)
        self.stride, self.padding = stride, padding

    def forward(self, xb: Float[torch.Tensor, "batch in_channels height width"]) -> Float[torch.Tensor, "batch out_channels new_height new_width"]:
        # TODO: complete the code.
        raise NotImplementedError("Optional exercise: MyStridedConv2d.forward")

In [ ]:
check_strided_conv(MyStridedConv2d)
print(MyStridedConv2d(1, 16, 5, stride=2, padding=2)(xb).shape)  # As torch.nn.Conv2d in Section 2.

## Optional B: depthwise convolutions ★★

Remember from the lecture that a convolution mixes information in two ways: across positions (spatial mixing) and across channels (channel mixing). A $1 \times 1$ convolution only mixes channels: it is the positionwise MLP we saw in the MLP lecture. A **depthwise convolution** does the opposite: each channel has its own spatial filter, and channels are never mixed. For a filter $K \sim (s, s, C)$ with $s = 2k + 1$:

$$Y_{i,j,a} = \sum_{u=-k}^{k}\sum_{v=-k}^{k} K_{u+k,\,v+k,\,a}\, X_{i+u,\,j+v,\,a} + b_a.$$

A depthwise convolution followed by a $1 \times 1$ one is a **depthwise-separable** convolution. In PyTorch, a depthwise convolution is `torch.nn.Conv2d(C, C, s, groups=C)`.

> **TODO** (★★): complete `MyDepthwiseConv2d` (stride 1). As in PyTorch, `weight` has shape `(channels, 1, kernel_size, kernel_size)` and `bias` shape `(channels,)`. Use He initialization with $d = s^2$ inputs per output.

Hint: the rows of `unfold` are ordered by channel first, so you can view its output as $(B, C, s^2, L)$. Each channel now has its own patches: you do not need a matrix product.

In [ ]:
class MyDepthwiseConv2d(torch.nn.Module):
    def __init__(self, channels: int, kernel_size: int, padding: int = 0):
        super().__init__()
        # TODO: complete the code.
        raise NotImplementedError("Optional exercise: MyDepthwiseConv2d.__init__")

    def forward(self, xb: Float[torch.Tensor, "batch channels height width"]) -> Float[torch.Tensor, "batch channels new_height new_width"]:
        # TODO: complete the code.
        raise NotImplementedError("Optional exercise: MyDepthwiseConv2d.forward")

In [ ]:
check_depthwise(MyDepthwiseConv2d)

A depthwise-separable layer is now just a composition of the two layers you wrote.

**Predict.** How many parameters does a standard $3 \times 3$ convolution from 64 to 128 channels have? And a separable one with the same input and output channels? Compute them by hand, then run the cell.

In [ ]:
class SeparableConv2d(torch.nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, padding: int = 0):
        super().__init__()
        self.depthwise = MyDepthwiseConv2d(in_channels, kernel_size, padding=padding)
        self.pointwise = MyConv2d(in_channels, out_channels, 1)

    def forward(self, xb):
        return self.pointwise(self.depthwise(xb))

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

print("Standard: ", count_parameters(MyConv2d(64, 128, 3, padding=1)))
print("Separable:", count_parameters(SeparableConv2d(64, 128, 3, padding=1)))
assert SeparableConv2d(64, 128, 3, padding=1)(torch.randn(2, 64, 8, 8)).shape == (2, 128, 8, 8)

## Optional C: translation equivariance ★★

A convolutional layer is *translation equivariant*: if we shift the input, the output shifts by the same amount. With zero-padding this cannot hold exactly near the borders, since something has to enter the image from outside. The function `shift` below moves a batch of images by `dy` rows and `dx` columns, filling the empty part with `fill`.

In [ ]:
def shift(x: Float[torch.Tensor, "batch channels height width"], dy: int, dx: int, fill: float = 0.0) -> Float[torch.Tensor, "batch channels height width"]:
    H, W = x.shape[-2:]
    out = torch.full_like(x, fill)
    out[..., max(dy, 0):H + min(dy, 0), max(dx, 0):W + min(dx, 0)] = \
        x[..., max(-dy, 0):H - max(dy, 0), max(-dx, 0):W - max(dx, 0)]
    return out

fig, axes = plt.subplots(1, 3, figsize=(6, 2))
for ax, (dy, dx) in zip(axes, [(0, 0), (3, 0), (-2, 4)]):
    ax.imshow(shift(xb[:1], dy, dx, fill=background)[0, 0], cmap="gray_r")
    ax.set_title(f"dy={dy}, dx={dx}")
    ax.axis("off")
plt.show()

> **TODO** (★★): complete `equivariance_error`, which shifts the input by `(dy, dx)`, applies `layer`, and compares the result with the shifted output of the original input. Both outputs are cropped by `border` pixels on each side before computing the largest absolute difference. Use the default `fill=0` in both shifts: it is the same value used by zero-padding. Assume that `layer` keeps the height and width of its input.

Then use it to answer the questions after the check.

In [ ]:
def equivariance_error(layer, x: Float[torch.Tensor, "batch channels height width"], dy: int, dx: int, border: int) -> float:
    # TODO: complete the code. Return a Python float.
    raise NotImplementedError("Optional exercise: equivariance_error")

In [ ]:
check_equivariance_error(equivariance_error)

Use `equivariance_error` on a batch of KMNIST images (for example `xb[:8]`) to answer:

1. For `cnn.conv1` (a trained 3×3 convolution with padding 1) and a shift of 2 pixels in both directions, what is the smallest `border` for which the error is zero (up to float precision)? Why that value?
2. Repeat with `lambda x: cnn.conv2(F.relu(cnn.conv1(x)))`, two stacked layers. How does the border change?
3. Max-pooling halves the resolution, so the function cannot be used directly. Instead, compare `pool(shift(x, d, d))` with `shift(pool(x), d // 2, d // 2)` for `d = 2` and `d = 1`, with `pool = MyMaxPool2d(2)`. What happens?

In [ ]:
images = xb[:8]
# Your experiments here.

What does this mean for classification? The cell below shifts the validation images by an increasing number of pixels (in a random direction, filling with background) and measures the accuracy of the two models.

**Predict.** Which model degrades faster?

In [ ]:
@torch.no_grad()
def shifted_accuracy(model, x, y, d, seed=0):
    model.eval()
    gen = torch.Generator().manual_seed(seed)
    signs = torch.randint(0, 2, (len(x), 2), generator=gen) * 2 - 1
    moved = torch.stack([shift(img[None], d * int(sy), d * int(sx), fill=background)[0]
                         for img, (sy, sx) in zip(x, signs)])
    return (model(moved).argmax(dim=1) == y).float().mean().item()

distances = range(0, 6)
results = {name: [shifted_accuracy(model, val_x, val_y, d) for d in distances]
           for name, model in [("CNN", cnn), ("MLP", mlp)]}
plt.figure(figsize=(5, 3.5))
for name, accuracies in results.items():
    plt.plot(distances, accuracies, "o-", label=name)
plt.xlabel("Shift (pixels in each direction)")
plt.ylabel("Validation accuracy")
plt.legend()
plt.show()

## Optional D: a locally-connected layer ★★★

Having full control over the convolution allows us to implement several interesting variations. A *locally-connected* layer is identical to the convolutional one, except that the weights are not tied: each sliding window is multiplied by a different kernel matrix.

> **TODO** (★★★): modify your implementation of `MyConv2d` to obtain a locally-connected variant (stride 1). It needs to know the input size, to create one kernel per window: the weight has shape `(L, out_channels, in_channels * kernel_size**2)`, with `L` the number of windows, and the bias has shape `(L, out_channels)`. This increases the number of parameters by a factor of `L`.

Hint: `torch.einsum` is the easiest way to multiply each window by its own matrix.

In [ ]:
class MyLocallyConnected2d(torch.nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, height: int, width: int, padding: int = 0):
        super().__init__()
        # TODO: complete the code.
        raise NotImplementedError("Optional exercise: MyLocallyConnected2d.__init__")

    def forward(self, xb: Float[torch.Tensor, "batch in_channels height width"]) -> Float[torch.Tensor, "batch out_channels new_height new_width"]:
        # TODO: complete the code.
        raise NotImplementedError("Optional exercise: MyLocallyConnected2d.forward")

In [ ]:
check_locally_connected(MyLocallyConnected2d, MyConv2d)
print("Parameters, convolution:       ", sum(p.numel() for p in MyConv2d(1, 16, 3, padding=1).parameters()))
print("Parameters, locally connected: ", sum(p.numel() for p in MyLocallyConnected2d(1, 16, 3, 28, 28, padding=1).parameters()))
fresh = MyLocallyConnected2d(1, 16, 3, 28, 28, padding=1)
print("Equivariance error (border 3): ", equivariance_error(fresh, xb[:8], 2, 2, 3))

The last line uses your function from Optional C: unlike a convolution, a locally-connected layer is not equivariant, even far from the borders. It has the same locality as a convolution but no weight sharing, with 784 times more parameters on KMNIST.

## Optional E: patches without copies ★★★

`unfold` copies every pixel into $s^2$ different columns. In PT01 we saw that a tensor is a view on a block of memory, described by its shape and its *strides* (how many elements to skip to move along each axis). With the right strides, we can describe all the patches of an image as a view, without copying anything.

> **TODO** (★★★): complete `patches_view`, which receives a (padded) batch `x` of shape $(B, C, H, W)$ and returns a view of shape $(B, C, H - s + 1, W - s + 1, s, s)$, where `out[b, c, i, j]` is the $s \times s$ patch with top-left corner $(i, j)$. Use [`as_strided`](https://docs.pytorch.org/docs/stable/generated/torch.as_strided.html) and `x.stride()`.

Hint: moving to the next patch to the right and moving to the next pixel to the right inside a patch skip the same number of elements in memory.

In [ ]:
def patches_view(x: Float[torch.Tensor, "batch channels height width"], kernel_size: int) -> Float[torch.Tensor, "batch channels rows cols kernel kernel"]:
    # TODO: complete the code.
    raise NotImplementedError("Optional exercise: patches_view")

In [ ]:
check_patches_view(patches_view)
view = patches_view(torch.randn(2, 3, 6, 7), 3)
print("Is the view contiguous?", view.is_contiguous())

A convolution is now an `einsum` over the view. The cell compares the memory used by `unfold` with the size of the input, and measures the time of three implementations on a batch of 64 images with 32 channels. Timings depend a lot on your hardware: we run each function a few times before measuring, and we report the median.

In [ ]:
import torch.utils.benchmark as benchmark

def einsum_conv(x, weight, bias, padding):
    view = patches_view(F.pad(x, (padding,) * 4), weight.shape[-1])
    return torch.einsum("bchwij,ocij->bohw", view, weight) + bias[:, None, None]

x = torch.randn(64, 32, 28, 28)
layer = MyConv2d(32, 64, 3, padding=1)
with torch.no_grad():
    torch.testing.assert_close(einsum_conv(x, layer.weight, layer.bias, 1), layer(x), rtol=1e-4, atol=1e-4)
    unfold_bytes = F.unfold(x, 3, padding=1).numel() * 4
print(f"Input: {x.numel() * 4 / 2**20:.1f} MiB, unfold: {unfold_bytes / 2**20:.1f} MiB")

functions = {
    "MyConv2d (unfold)": lambda: layer(x),
    "einsum on a view": lambda: einsum_conv(x, layer.weight, layer.bias, 1),
    "F.conv2d": lambda: F.conv2d(x, layer.weight, layer.bias, padding=1),
}
with torch.no_grad():
    for name, function in functions.items():
        for _ in range(3):
            function()  # Warm-up.
        timer = benchmark.Timer(stmt="function()", globals={"function": function})
        print(f"{name:<18}: {timer.blocked_autorange(min_run_time=0.5).median * 1e3:6.1f} ms")

**Discuss.** The view itself costs no memory. Does that mean the `einsum` never copies? (Try `torch.einsum` on a non-contiguous input and think about how it is turned into a matrix product.) Why is `F.conv2d` still the fastest?

## Optional F: measuring receptive fields ★★

The *receptive field* of a unit is the set of input pixels that can affect it. In the lecture we computed it by hand. Here we measure it with autograd: the gradient of one output unit with respect to the input is nonzero exactly on the pixels the unit depends on.

> **TODO** (★★): complete `receptive_field(network, size)`. Run `network` on a random input of shape `(1, 1, size, size)` that requires gradients, take the output unit of channel 0 at the center of the output (`out[0, 0, h // 2, w // 2]`), compute its gradient with respect to the input, and return the height and width of the smallest rectangle containing all its nonzero entries.

To avoid gradients that are zero by chance (after a ReLU or a max-pooling), use networks made of convolutions only, with all-positive weights.

In [ ]:
def receptive_field(network, size: int = 64) -> tuple[int, int]:
    # TODO: complete the code.
    raise NotImplementedError("Optional exercise: receptive_field")

In [ ]:
check_receptive_field(receptive_field)

def positive_convs(*specs):  # (kernel, stride) for each layer.
    layers = [torch.nn.Conv2d(1, 1, k, stride=stride, padding=k // 2) for k, stride in specs]
    with torch.no_grad():
        for layer in layers:
            layer.weight.uniform_(0.1, 1.)
    return torch.nn.Sequential(*layers)

for specs in [[(3, 1)] * 2, [(3, 1)] * 3, [(3, 1)] * 5, [(3, 2), (3, 1)], [(3, 2), (3, 2), (3, 1)], [(5, 1), (3, 1)]]:
    print(f"{specs}: receptive field {receptive_field(positive_convs(*specs))}")

**Discuss.** How does the receptive field grow when you stack 3×3 layers with stride 1? What changes after a layer with stride 2? Write a formula, and use it to compute the receptive field of the last convolution of `MyCNN` (treating each max-pooling as a 2×2 layer with stride 2).

## Optional G: causal convolutions ★★

*Together with the second half of the CNN lecture.* Convolutions also work on sequences: a 1D convolution slides a filter of size $k$ along the time axis of an input $(B, C, T)$. For next-token prediction, the output at time $t$ must not see the inputs after $t$, otherwise the model can simply read the answer. A **causal** convolution pads $k - 1$ zeros on the left only, so that each output looks at the current step and the $k - 1$ previous ones.

> **TODO** (★★): complete `causal_conv1d(x, weight, bias)`, with `weight` of shape $(C', C, k)$ as in `F.conv1d`. Use `F.pad` and the tensor method [`Tensor.unfold`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.unfold.html) (not the same as `F.unfold`!) to extract the windows, then an `einsum`.

In [ ]:
def causal_conv1d(x: Float[torch.Tensor, "batch channels time"], weight: Float[torch.Tensor, "out_channels channels kernel"], bias: Float[torch.Tensor, "out_channels"]) -> Float[torch.Tensor, "batch out_channels time"]:
    # TODO: complete the code.
    raise NotImplementedError("Optional exercise: causal_conv1d")

In [ ]:
check_causal_conv1d(causal_conv1d)

# Which inputs does each output depend on? Row t of the Jacobian is output t.
gen = torch.Generator().manual_seed(10)
def dependence(function, T=12):
    jac = torch.autograd.functional.jacobian(lambda s: function(s[None, None])[0, 0], torch.randn(T, generator=gen))
    return (jac != 0).float()

w1, b1 = torch.rand(1, 1, 5, generator=gen) + 0.1, torch.zeros(1)
causal = dependence(lambda s: causal_conv1d(s, w1, b1))
centered = dependence(lambda s: F.conv1d(s, w1, b1, padding=2))
fig, axes = plt.subplots(1, 2, figsize=(7, 3.4))
for ax, matrix, title in zip(axes, [causal, centered], ["Causal", "Centered (padding 2)"]):
    ax.imshow(matrix, cmap="Blues")
    ax.set(title=title, xlabel="Input time step", ylabel="Output time step")
fig.tight_layout()
plt.show()

**Discuss.** In the centered convolution, output $t$ depends on inputs $t+1$ and $t+2$. If we trained it to predict the next token, why would the training loss be misleadingly low? How does the receptive field of a stack of causal layers grow, and how could you make it grow faster without more layers?

## Sources

- Simone Scardapane, original Colab lab: [Convolutional neural networks (from scratch)](https://colab.research.google.com/drive/1Qu_xTcZHLbyVXIbAcuduRCRO3zqJM488).
- NNDS 2026 lecture on convolutional neural networks.
- [PT01](PT01_Introduction_to_PyTorch.ipynb) (views and strides, autograd) and [PT02](PT02_Logistic_regression.ipynb) (the training loop).
- KMNIST: Clanuwat et al. (2018), [Deep learning for classical Japanese literature](https://arxiv.org/abs/1812.01718), dataset at [github.com/rois-codh/kmnist](https://github.com/rois-codh/kmnist) (CC BY-SA 4.0).
- Occlusion maps: Zeiler and Fergus (2014), [Visualizing and understanding convolutional networks](https://arxiv.org/abs/1311.2901), ECCV.
- PyTorch documentation for [`F.unfold`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.unfold.html), [`Tensor.as_strided`](https://docs.pytorch.org/docs/stable/generated/torch.as_strided.html) and [`torch.utils.benchmark`](https://docs.pytorch.org/docs/stable/benchmark_utils.html).